# Retrieve Candidates Step-by-Step

This notebook walks through `retrieve_candidates` one step per cell.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession

from taxomind.utils import embedding_utils
from taxomind.pipelines.inference.nodes import build_retrieval_index

project_path = Path.cwd()
if not (project_path / "conf").exists():
    project_path = project_path.parent
bootstrap_project(project_path)


In [ ]:
taxonomy_key = "ISCO"
query_text = "addressing customer issues call center"
retrieval_k = 20
beam_count = 3


In [ ]:
with KedroSession.create(project_path=project_path) as session:
    context = session.load_context()
    params = context.params
    taxonomy_partitions = context.catalog.load("taxonomy_index")

taxonomy_df = taxonomy_partitions[taxonomy_key]()

model_name = params.get("model_name")
cache_dir = params.get("embedding", {}).get("cache_dir")
local_files_only = params.get("embedding", {}).get("local_files_only", False)
query_prefix = params.get("embedding_prefix", {}).get("query")

embedding_model = embedding_utils.load_embedding_model(
    model_name,
    cache_dir=cache_dir,
    local_files_only=local_files_only,
)

retrieval_index = build_retrieval_index(taxonomy_df)


In [ ]:
query_embeddings, _ = embedding_utils.encode_texts(
    embedding_model,
    [query_text],
    embed_all=True,
    input_prefix=query_prefix,
    batch_size=32,
    show_progress_bar=False,
)
query_embedding = query_embeddings[0]
query_embedding.shape


In [ ]:
# Step 0: Extract index components
label_embeddings = retrieval_index["embeddings"]
codes = retrieval_index["codes"]
code_to_parent = retrieval_index["code_to_parent"]
code_to_children = retrieval_index.get("code_to_children", {})
(len(codes), label_embeddings.shape)


In [ ]:
# Step 1: Retrieve top-K by label similarity
similarities = np.dot(label_embeddings, query_embedding)
top_k_indices = np.argsort(similarities)[::-1][:retrieval_k]
retrieved_codes = [codes[i] for i in top_k_indices]
retrieved_scores = {codes[i]: float(similarities[i]) for i in top_k_indices}
R = set(retrieved_codes)
retrieved_codes

In [ ]:
retrieved_scores

In [ ]:
taxonomy_df[taxonomy_df["code"].isin(retrieved_codes)]

In [ ]:
# Step 2: Ancestor closure A
A = set()
for code in R:
    current = code
    while current and current != "__root__":
        parent = code_to_parent.get(current)
        if parent and parent != "__root__":
            A.add(parent)
        current = parent
sorted(A)#[:20]


In [ ]:
# Step 3: Sibling completion S = children(A ∪ {__root__})
S = set()
parent_set = set(A)
parent_set.add("__root__")
for parent in parent_set:
    for child in code_to_children.get(parent, []):
        S.add(child)
len(S),S

In [ ]:
# Step 4: Candidate set V = R ∪ A ∪ S
V = R | A | S
(len(R), len(A), len(S), len(V))
V

In [ ]:
# Step 5: Aggregate evidence by root (L1)
def get_root_ancestor(code: str) -> str:
    current = code
    path = [current]
    while current and current != "__root__":
        parent = code_to_parent.get(current)
        if parent == "__root__":
            break
        path.append(parent)
        current = parent
    for node in reversed(path):
        if code_to_parent.get(node) == "__root__":
            return node
    return path[0]

root_evidence = {}
for code in retrieved_codes:
    root = get_root_ancestor(code)
    root_evidence.setdefault(root, []).append(retrieved_scores[code])
{root: sum(scores) for root, scores in root_evidence.items()},root_evidence


In [ ]:
{root: sum(scores)/len(scores) for root, scores in root_evidence.items()}

In [ ]:
x = {root: sum(scores) for root, scores in root_evidence.items()}
x = pd.DataFrame.from_dict(x,orient="index", columns=["score"])
x['score_norm'] = x['score']/ x['score'].sum()
x

In [ ]:
# Step 6: Select top-B root beams
beam_roots_ranked = sorted(
    root_evidence.items(),
    key=lambda x: sum(x[1]),
    reverse=True,
)[:beam_count]
beam_roots = [root for root, _ in beam_roots_ranked]
beam_roots


In [ ]:
# Final output (same structure as retrieve_candidates)
result = {
    "retrieved_codes": retrieved_codes,
    "retrieved_scores": retrieved_scores,
    "V_codes": V,
    "ancestors": A,
    "siblings": S,
    "beam_roots": beam_roots,
    "root_evidence": root_evidence,
}
{
    "retrieved": len(result["retrieved_codes"]),
    "ancestors": len(result["ancestors"]),
    "siblings": len(result["siblings"]),
    "V_codes": len(result["V_codes"]),
    "beam_roots": result["beam_roots"],
}


# Batch Route Top-Down (step-by-step)

This section mirrors `batch_route_topdown` with one step per cell.


In [ ]:
from taxomind.pipelines.inference.nodes import (
    load_taxonomy_graph,
    prepare_scoring_views,
    batch_retrieve_candidates,
    batch_route_topdown,
)


In [ ]:
# Step 1: Build a batch of queries
batch_queries = [
    "addressing customer issues call center",
]
queries_df = pd.DataFrame({"query_id": range(len(batch_queries)), "text": batch_queries})
queries_df


In [ ]:
# Step 2: Embed queries (batch)
query_embeddings, indices = embedding_utils.encode_texts(
    embedding_model,
    queries_df["text"].tolist(),
    embed_all=True,
    input_prefix=query_prefix,
    batch_size=32,
    show_progress_bar=False,
)
queries_df = queries_df.copy()
queries_df["embedding"] = list(query_embeddings)
queries_df.head()


In [ ]:
# Step 3: Batch retrieve candidates
candidates_df = batch_retrieve_candidates(
    queries_df=queries_df,
    retrieval_index=retrieval_index,
    retrieval_k=retrieval_k,
    beam_count=beam_count,
)
candidates_df[["query_id", "candidates"]].head()


In [ ]:
# Step 4: Prepare scoring views + taxonomy graph
use_updated_evidence = params.get("inference", {}).get("use_updated_evidence", False)
scoring_views = prepare_scoring_views(taxonomy_df, use_updated_evidence=use_updated_evidence)
taxonomy_graph = load_taxonomy_graph(taxonomy_df)
(len(taxonomy_graph), len(scoring_views.get("code_to_label", {})))

In [ ]:
# Step 5: Batch route top-down
inference_params = params.get("inference", {})
routing_df = batch_route_topdown(
    candidates_df=candidates_df,
    scoring_views=scoring_views,
    taxonomy_graph=taxonomy_graph,
    min_descent_gap=inference_params.get("min_descent_gap", 0.05),
    parent_veto_margin=inference_params.get("parent_veto_margin", 0.05),
    enable_parent_veto=inference_params.get("enable_parent_veto", True),
    evidence_tau=inference_params.get("evidence_tau", 10.0),
    evidence_max_beta=inference_params.get("evidence_max_beta", 0.8),
    short_query_tokens=inference_params.get("short_query_tokens", 2),
    max_depth=inference_params.get("max_depth", None),
)
routing_df[["query_id", "routing_result"]].head()


In [ ]:
# Step 6: Inspect routing decisions
routing_preview = routing_df[["query_id", "text", "routing_result"]].copy()
routing_preview["predicted_code"] = routing_preview["routing_result"].map(lambda r: r.get("predicted_code"))
routing_preview["stopping_reason"] = routing_preview["routing_result"].map(lambda r: r.get("stopping_reason"))
routing_preview[["query_id", "text", "predicted_code", "stopping_reason"]]
